# Preparation

In [ ]:
import os
import sys

# Go up one directory level to the project root folder
project_root = os.path.abspath(os.path.join(os.getcwd(), ".."))
print(project_root)

if project_root not in sys.path:
    sys.path.append(project_root)

# Now you can successfully import from your module folder


In [ ]:
from module.DepthConstSet import DepthConstSet
from module.NyuDatasetV2 import *
from module.MLModel import *
from module.Training import train_model, plot_metrics, test_model

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torchvision import transforms
import matplotlib.pyplot as plt # For visualization


In [ ]:
# Ensure reproducibility for consistent results
torch.manual_seed(42)
np.random.seed(42)

# Define the path to your NYU Depth V2 labeled dataset (.mat file).
# You MUST replace this with the actual path on your system.
# The dataset can typically be downloaded from the official NYU Depth V2 website
# or other academic sources.
NYU_DATASET_PATH = 'nyu_depth_v2_labeled.mat' 
nyuv2_path = project_root + "/module/nyuv2_python_toolkit_master/NYUv2"

# Define the number of classes for segmentation.
# For NYU Depth V2, a common setup uses 40 semantic classes + 1 for unlabeled/background (label 0).
# So, total classes = 41.
NUM_CLASSES = 13
BATCH_SIZE = 8
LR = 2e-2
WEIGHT_DECAY = 5e-4

# Define the target image size for resizing. NYU images are 480x640.
# Downsampling can speed up training.
# IMAGE_SIZE = (240, 320) 
# IMAGE_SIZE = (480, 640) 
IMAGE_SIZE = (480, 480) 
pseudo_depth_models:tuple = [DepthConstSet().marigold,DepthConstSet().midas]

EPOCH = 50

In [ ]:
# Set the device for training (GPU if available, otherwise CPU)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
# device = torch.device("cpu")
print(f"Using device: {device}")
print(f"torch ver: {torch.version.cuda}")

# Load dataset

## Set Dataloader

In [ ]:
train_loader, val_loader, test_loader = get_train_val_dataloaders(
    dataset_path=nyuv2_path,
    batch_size=BATCH_SIZE,         # Small batch size for demonstration; increase for real training
    image_size=IMAGE_SIZE,
    transform_depth = True,
    auto_augment=False,
    transform_depth_dict=pseudo_depth_models
)

In [ ]:
first_batch = next(iter(train_loader))
# image, (depth, label) = first_batch
(img, depthDict, GT) = first_batch

In [ ]:
depthDict["raw"].shape

In [ ]:
depthDict.keys()

In [ ]:
depthDict[DepthConstSet().raw].shape

In [ ]:
GT.shape

In [ ]:
depthImg = depthDict[DepthConstSet().raw][0]
depthImg.shape

# Visualize dataset

In [ ]:
# --- Visualize a batch from the train_loader ---
print("\nVisualizing a batch from the training DataLoader...")
# Get one batch
# first_batch = next(iter(train_loader))


In [ ]:
# Categories
# train_loader.dataset.dataset.__getLabelList__()

In [ ]:
(rgb_images, depth_maps, labels) = first_batch

In [ ]:
# To denormalize RGB images correctly, we need access to the NyuDataset instance's
# mean and std. We can get this from the `dataset` attribute of the DataLoader's
# `dataset` attribute (which is a Subset object from random_split).
# The `dataset` attribute of the `Subset` object holds the original `NyuDataset`.

visualize_batch(first_batch, NUM_CLASSES, transform_depth=False, transform_hha=True)
print("Visualization complete. A plot window should have appeared.")
# --- End Visualization ---


# Model

In [ ]:
from module.RGBD_Semantic_Segmentation_PyTorch.model.SA_Gate.network import DeepLab
# from module.RGBD_Semantic_Segmentation_PyTorch.furnace.seg_opr.sync_bn import BatchNorm2d
# from apex.parallel import DistributedDataParallel, SyncBatchNorm

import torch.nn as nn
# from seg_opr.sync_bn import DataParallelModel, Reduce, BatchNorm2d
BatchNorm2d = nn.SyncBatchNorm

modelName = "SA_Gate"
# modelRaw = SegFormerDepth(num_classes=NUM_CLASSES, image_size=IMAGE_SIZE)
# config network and criterion
# criterion = nn.CrossEntropyLoss(reduction='mean', ignore_index=255)
criterion = nn.CrossEntropyLoss(reduction='mean')
pretrainModelPath = project_root + "/module/RGBD_Semantic_Segmentation_PyTorch/DATA/pytorch_weight/resnet101_v1c.pth"
modelRaw = DeepLab(NUM_CLASSES, criterion=criterion,
                pretrained_model=pretrainModelPath,
                norm_layer=BatchNorm2d)
modelRaw.eval()

In [ ]:
from module.RGBD_Semantic_Segmentation_PyTorch.furnace.utils.init_func import init_weight, group_weight

bn_eps = 1e-5
bn_momentum = 0.1
init_weight(modelRaw.business_layer, nn.init.kaiming_normal_,
            BatchNorm2d, bn_eps, bn_momentum,
            mode='fan_in', nonlinearity='relu')

params_list = []
params_list = group_weight(params_list, modelRaw.backbone,
                            BatchNorm2d, LR)
for module in modelRaw.business_layer:
    params_list = group_weight(params_list, module, BatchNorm2d,
                                LR)
    
optimizer = torch.optim.SGD(params_list,
                        lr=LR,
                        momentum=0.9,
                        weight_decay=WEIGHT_DECAY)

In [ ]:
xImg = torch.randn((BATCH_SIZE, 3, IMAGE_SIZE[0], IMAGE_SIZE[1]))
xImg.shape
yhha = torch.randn((BATCH_SIZE, 3, IMAGE_SIZE[0], IMAGE_SIZE[1]))
yhha.shape
zGt = torch.randn((BATCH_SIZE, 13, IMAGE_SIZE[0], IMAGE_SIZE[1]))
zGt.shape

In [ ]:
## training example
with torch.no_grad():
    loss, loss_aux, pred = modelRaw(xImg, yhha, zGt)
    
    aux_rate = 0.2
    tot_loss = loss + loss_aux * aux_rate
# logits = outputs.logits  # [B, num_classes, h, w]


In [ ]:
print(f"x shape: {xImg.shape}")
print(f"pred shape: {pred.shape}")
predShape=torch.Size([BATCH_SIZE, NUM_CLASSES, IMAGE_SIZE[0], IMAGE_SIZE[1]])
print(f"expected shape: {predShape}")
assert pred.shape == predShape

In [ ]:
## testing example
with torch.no_grad():
    predVal = modelRaw(xImg, yhha)
    


In [ ]:
print(f"x shape: {xImg.shape}")
print(f"pred shape: {predVal.shape}")
predShape=torch.Size([BATCH_SIZE, NUM_CLASSES, IMAGE_SIZE[0], IMAGE_SIZE[1]])
print(f"expected shape: {predShape}")
assert predVal.shape == predShape

# Training Depth Depth

## Training

In [ ]:
depthToTest = "PDAM_2PD_1"
model_save_path = modelName + "_depth_segmentation_"+ depthToTest + ".pth" # Define path for saved model# NUM_CLASSES=14
train_losses, train_accuracies, train_mious, val_losses, val_accuracies, val_mious = train_model(
    model=modelRaw,
    train_dataloader=train_loader,
    val_dataloader=val_loader, 
    num_classes=NUM_CLASSES,
    epochs=EPOCH, # Small number of epochs for demonstration; increase for actual training
    learning_rate=LR,
    weight_decay=WEIGHT_DECAY,
    device=device,
    model_save_path=model_save_path,
    use_hha = True,
    criterion = criterion,
    optimizer = optimizer,
    use_scheduler=False,
    use_PDAM=True,
    pseudo_depth_models=pseudo_depth_models
    # depth_all_zeros = True,
    # use_logits = True
)

In [ ]:
# Plot the collected metrics after training
print("\n--- Plotting Training and Validation Metrics ---")
plot_metrics(train_losses, train_accuracies, train_mious, 
                val_losses, val_accuracies, val_mious, epochs=EPOCH)
print("Metric plots generated.")

## Testing

In [ ]:
# --- Test the trained model ---
print("\n--- Running Test Model ---")
# You can use val_loader as a proxy for a test set for this example
test_model(
    model=modelRaw,
    test_dataloader=test_loader, # Using test loader for demonstration
    num_classes=NUM_CLASSES,
    device=device,
    model_load_path=model_save_path,
    visualize_samples=BATCH_SIZE, # Visualize 2 samples from the test set
    use_hha = True,
    use_PDAM=True,    
    pseudo_depth_models=pseudo_depth_models
    # use_logits = True
)
